# Tokenization


In [1]:
from __future__ import annotations

import math
import re
from collections import Counter
from functools import lru_cache
from pathlib import Path
from pprint import pprint

import tiktoken
from tokenizers import Regex
from tokenizers import Tokenizer as HFTokenizer
from tokenizers import decoders, pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer

LECTURE_NOTE_TITLE = 'Tokenization'

sample_text = """Hello, world. This, is a test.
It's the last he painted, you know.
In the sunlit terraces of someunknownPlace, we saw café prices, code like foo_bar(), and emoji 😊.
"""

comparison_text = "Toronto 2026-04-06: foo_bar() costs 19.99 dollars, and café 😊 is still open."

print(f'Lecture note: {LECTURE_NOTE_TITLE}')
print(sample_text)
print(comparison_text)


Lecture note: Tokenization
Hello, world. This, is a test.
It's the last he painted, you know.
In the sunlit terraces of someunknownPlace, we saw café prices, code like foo_bar(), and emoji 😊.

Toronto 2026-04-06: foo_bar() costs 19.99 dollars, and café 😊 is still open.


<a id="word-level-tokenizer-and-the-oov-failure"></a>
## Word-level tokenizer and the OOV failure


In [2]:
def word_level_tokenize(text: str) -> list[str]:
    return text.split()


def character_level_tokenize(text: str) -> list[str]:
    return list(text)


text = "hello world"
word_tokens = word_level_tokenize(text)
char_tokens = character_level_tokenize(text)

print("word tokens:", word_tokens)
print("character tokens:", char_tokens)
print("word token count:", len(word_tokens))
print("character token count:", len(char_tokens))


word tokens: ['hello', 'world']
character tokens: ['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd']
word token count: 2
character token count: 11


In [3]:
for text in [
    "someunknownPlace",
    "foo_bar()",
    "😊 café",
]:
    print("=" * 80)
    print(text)
    print("word-level:", word_level_tokenize(text))
    print("character-level:", character_level_tokenize(text))


someunknownPlace
word-level: ['someunknownPlace']
character-level: ['s', 'o', 'm', 'e', 'u', 'n', 'k', 'n', 'o', 'w', 'n', 'P', 'l', 'a', 'c', 'e']
foo_bar()
word-level: ['foo_bar()']
character-level: ['f', 'o', 'o', '_', 'b', 'a', 'r', '(', ')']
😊 café
word-level: ['😊', 'café']
character-level: ['😊', ' ', 'c', 'a', 'f', 'é']


<a id="regex-tokenizer-and-fallback-tokens"></a>
## Regex tokenizer and fallback tokens


In [4]:
rasbt_pattern = r"([,.:;?_!\"()']|--|\s)"


def rasbt_split(text: str) -> list[str]:
    pieces = re.split(rasbt_pattern, text)
    return [item.strip() for item in pieces if item.strip()]


basic_text = "Hello, world. This, is a test."
pprint(rasbt_split(basic_text))
pprint(rasbt_split(comparison_text))


['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']
['Toronto',
 '2026-04-06',
 ':',
 'foo',
 '_',
 'bar',
 '(',
 ')',
 'costs',
 '19',
 '.',
 '99',
 'dollars',
 ',',
 'and',
 'café',
 '😊',
 'is',
 'still',
 'open',
 '.']


In [5]:
rasbt_tokens = rasbt_split(sample_text)
rasbt_vocab = {token: idx for idx, token in enumerate(sorted(set(rasbt_tokens)))}


class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        return [self.str_to_int[s] for s in rasbt_split(text)]

    def decode(self, ids: list[int]) -> str:
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r"\s+([,.:;?!\"()'])", r"\1", text)


simple_v1 = SimpleTokenizerV1(rasbt_vocab)
print(simple_v1.encode("It's the last he painted, you know."))
print(simple_v1.decode(simple_v1.encode("It's the last he painted, you know.")))


[7, 0, 25, 31, 20, 17, 23, 3, 34, 19, 4]
It' s the last he painted, you know.


In [6]:
try:
    simple_v1.encode("Hello, do you like tea?")
except KeyError as exc:
    print("OOV error:", exc)

rasbt_vocab_v2 = {token: idx for idx, token in enumerate(sorted(set(rasbt_tokens)) + ["<|endoftext|>", "<|unk|>"])}


class SimpleTokenizerV2:
    def __init__(self, vocab: dict[str, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        pieces = rasbt_split(text)
        pieces = [piece if piece in self.str_to_int else "<|unk|>" for piece in pieces]
        return [self.str_to_int[s] for s in pieces]

    def decode(self, ids: list[int]) -> str:
        text = " ".join(self.int_to_str[i] for i in ids)
        return re.sub(r"\s+([,.:;?!\"()'])", r"\1", text)


simple_v2 = SimpleTokenizerV2(rasbt_vocab_v2)
print(simple_v2.encode(comparison_text))
print(simple_v2.decode(simple_v2.encode(comparison_text)))


OOV error: 'do'
[37, 37, 37, 16, 9, 12, 1, 2, 37, 37, 4, 37, 37, 3, 11, 13, 35, 18, 37, 37, 4]
<|unk|> <|unk|> <|unk|> foo _ bar() <|unk|> <|unk|>. <|unk|> <|unk|>, and café 😊 is <|unk|> <|unk|>.


<a id="bpe-merges-on-a-micro-corpus"></a>
## BPE merges on a micro-corpus


In [7]:
toy_vocab = {
    "l o w </w>": 5,
    "l o w e r </w>": 2,
    "n e w e s t </w>": 6,
    "w i d e s t </w>": 3,
}


def get_pair_stats(vocab_with_counts: dict[str, int]) -> Counter:
    stats = Counter()
    for word, freq in vocab_with_counts.items():
        symbols = word.split()
        for pair in zip(symbols, symbols[1:]):
            stats[pair] += freq
    return stats


def merge_pair(pair: tuple[str, str], vocab_with_counts: dict[str, int]) -> dict[str, int]:
    merged = {}
    pattern = re.compile(rf"(?<!\S){re.escape(' '.join(pair))}(?!\S)")
    replacement = ''.join(pair)
    for word, freq in vocab_with_counts.items():
        merged[pattern.sub(replacement, word)] = freq
    return merged


current_vocab = toy_vocab.copy()
for step in range(6):
    stats = get_pair_stats(current_vocab)
    best_pair, best_count = stats.most_common(1)[0]
    print(f"step {step + 1}: merge {best_pair} count={best_count}")
    current_vocab = merge_pair(best_pair, current_vocab)
    pprint(current_vocab)
    print()


step 1: merge ('e', 's') count=9
{'l o w </w>': 5,
 'l o w e r </w>': 2,
 'n e w es t </w>': 6,
 'w i d es t </w>': 3}

step 2: merge ('es', 't') count=9
{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est </w>': 6, 'w i d est </w>': 3}

step 3: merge ('est', '</w>') count=9
{'l o w </w>': 5, 'l o w e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}

step 4: merge ('l', 'o') count=7
{'lo w </w>': 5, 'lo w e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}

step 5: merge ('lo', 'w') count=7
{'low </w>': 5, 'low e r </w>': 2, 'n e w est</w>': 6, 'w i d est</w>': 3}

step 6: merge ('n', 'e') count=6
{'low </w>': 5, 'low e r </w>': 2, 'ne w est</w>': 6, 'w i d est</w>': 3}



In [8]:
gpt2 = tiktoken.get_encoding("gpt2")


def render_tiktoken_piece(token_id: int) -> str:
    raw = gpt2.decode_single_token_bytes(token_id)
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return repr(raw)


def inspect_tiktoken(text: str) -> dict[str, object]:
    ids = gpt2.encode(text, allowed_special={"<|endoftext|>"})
    pieces = [render_tiktoken_piece(token_id) for token_id in ids]
    piece_bytes = [list(gpt2.decode_single_token_bytes(token_id)) for token_id in ids]
    return {
        "ids": ids,
        "pieces": pieces,
        "piece_bytes": piece_bytes,
        "count": len(ids),
    }


for text in [
    "In the sunlit terraces of someunknownPlace.",
    comparison_text,
]:
    print("=" * 100)
    print(text)
    pprint(inspect_tiktoken(text))


In the sunlit terraces of someunknownPlace.
{'count': 11,
 'ids': [818, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13],
 'pieces': ['In',
            ' the',
            ' sun',
            'lit',
            ' terr',
            'aces',
            ' of',
            ' some',
            'unknown',
            'Place',
            '.']}
Toronto 2026-04-06: foo_bar() costs 19.99 dollars, and café 😊 is still open.
{'count': 26,
 'ids': [31359,
         1160,
         2075,
         12,
         3023,
         12,
         3312,
         25,
         22944,
         62,
         5657,
         3419,
         3484,
         678,
         13,
         2079,
         5054,
         11,
         290,
         40304,
         30325,
         232,
         318,
         991,
         1280,
         13],
 'pieces': ['Toronto',
            ' 20',
            '26',
            '-',
            '04',
            '-',
            '06',
            ':',
            ' foo',
            '_'

<a id="bytes-versus-tokens-on-the-same-string"></a>
## Bytes versus tokens on the same string

This section compares UTF-8 byte length, character length, and tokenizer output on the same inputs so you can see why byte-level coverage and token count are not the same thing.


In [ ]:
def inspect_bytes_vs_tokens(text: str) -> dict[str, object]:
    utf8_bytes = list(text.encode("utf-8"))
    token_view = inspect_tiktoken(text)
    return {
        "text": text,
        "character_count": len(text),
        "utf8_byte_count": len(utf8_bytes),
        "utf8_bytes": utf8_bytes,
        "token_count": token_view["count"],
        "token_pieces": token_view["pieces"],
    }


for text in [
    "hello",
    "café",
    "😊",
    "foo_bar()",
    "Hello 😊 café",
]:
    print("=" * 100)
    pprint(inspect_bytes_vs_tokens(text))


<a id="wordpiece-longest-match-decoding-rule"></a>
## WordPiece longest-match decoding rule


In [9]:
wordpiece_vocab = {
    "[UNK]",
    "the",
    "sun",
    "##lit",
    "terr",
    "##aces",
    "some",
    "##unknown",
    "##place",
    "place",
}


def wordpiece_tokenize_word(word: str, vocab: set[str]) -> list[str]:
    word = word.lower()
    start = 0
    pieces = []
    while start < len(word):
        end = len(word)
        current = None
        while end > start:
            piece = word[start:end]
            if start > 0:
                piece = "##" + piece
            if piece in vocab:
                current = piece
                break
            end -= 1
        if current is None:
            return ["[UNK]"]
        pieces.append(current)
        start = end
    return pieces


for word in ["sunlit", "terraces", "someunknownplace", "unseenword"]:
    print(word, "->", wordpiece_tokenize_word(word, wordpiece_vocab))


sunlit -> ['sun', '##lit']
terraces -> ['terr', '##aces']
someunknownplace -> ['some', '##unknown', '##place']
unseenword -> ['[UNK]']


<a id="sentencepiece-intuition-and-whitespace-markers"></a>
## SentencePiece intuition and whitespace markers


In [10]:
# This is a toy unigram tokenizer to show the core idea behind SentencePiece-style scoring.
# It is not the real SentencePiece library.

unigram_scores = {
    "▁": -0.1,
    "▁token": -0.2,
    "ization": -0.4,
    "▁tokenization": -0.05,
    "▁works": -0.2,
    "work": -0.5,
    "s": -0.3,
    "▁for": -0.2,
    "▁café": -0.2,
    "▁emoji": -0.2,
    "😊": -0.2,
}


def sentencepiece_pretokenize(text: str) -> str:
    return "▁" + text.replace(" ", "▁")


def unigram_segment(text: str, scores: dict[str, float]) -> tuple[float, list[str]]:
    @lru_cache(maxsize=None)
    def best(i: int):
        if i == len(text):
            return 0.0, []
        best_score = -float("inf")
        best_path = None
        for piece, score in scores.items():
            if text.startswith(piece, i):
                rest_score, rest_path = best(i + len(piece))
                total = score + rest_score
                if total > best_score:
                    best_score = total
                    best_path = [piece] + rest_path
        if best_path is None:
            return -float("inf"), [text[i]]
        return best_score, best_path
    return best(0)


sp_text = sentencepiece_pretokenize("tokenization works")
score, pieces = unigram_segment(sp_text, unigram_scores)
print("pretokenized:", sp_text)
print("pieces:", pieces)
print("score:", score)


pretokenized: ▁tokenization▁works
pieces: ['▁tokenization', '▁works']
score: -0.25


<a id="nanochat-style-gpt-tokenizer-design"></a>
## Nanochat-style GPT tokenizer design


In [11]:
SPECIAL_TOKENS = [
    "<|bos|>",
    "<|user_start|>",
    "<|user_end|>",
    "<|assistant_start|>",
    "<|assistant_end|>",
    "<|python_start|>",
    "<|python_end|>",
    "<|output_start|>",
    "<|output_end|>",
]

SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""


def build_nanochat_pretokenizer():
    split_regex = Regex(SPLIT_PATTERN)
    return pre_tokenizers.Sequence(
        [
            pre_tokenizers.Split(pattern=split_regex, behavior="isolated", invert=False),
            pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
        ]
    )


nanochat_pretok = build_nanochat_pretokenizer()
pprint(nanochat_pretok.pre_tokenize_str(comparison_text))


[('Toronto', (0, 7)),
 ('Ġ', (7, 8)),
 ('20', (8, 10)),
 ('26', (10, 12)),
 ('-', (12, 13)),
 ('04', (13, 15)),
 ('-', (15, 16)),
 ('06', (16, 18)),
 (':', (18, 19)),
 ('Ġfoo', (19, 23)),
 ('_bar', (23, 27)),
 ('()', (27, 29)),
 ('Ġcosts', (29, 35)),
 ('Ġ', (35, 36)),
 ('19', (36, 38)),
 ('.', (38, 39)),
 ('99', (39, 41)),
 ('Ġdollars', (41, 49)),
 (',', (49, 50)),
 ('Ġand', (50, 54)),
 ('ĠcafÃ©', (54, 59)),
 ('ĠðŁĺĬ', (59, 61)),
 ('Ġis', (61, 64)),
 ('Ġstill', (64, 70)),
 ('Ġopen', (70, 75)),
 ('.', (75, 76))]


In [12]:
def train_nanochat_style_tokenizer(text_iterator, vocab_size: int = 320) -> HFTokenizer:
    tokenizer = HFTokenizer(BPE(byte_fallback=True, unk_token=None, fuse_unk=False))
    tokenizer.normalizer = None
    tokenizer.pre_tokenizer = build_nanochat_pretokenizer()
    tokenizer.decoder = decoders.ByteLevel()
    trainer = BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=1,
        show_progress=False,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        special_tokens=SPECIAL_TOKENS,
    )
    tokenizer.train_from_iterator(text_iterator, trainer)
    return tokenizer


def inspect_hf_tokenizer(tokenizer: HFTokenizer, text: str) -> dict[str, object]:
    encoded = tokenizer.encode(text, add_special_tokens=False)
    return {
        "ids": encoded.ids,
        "tokens": encoded.tokens,
        "count": len(encoded.ids),
        "decoded": tokenizer.decode(encoded.ids, skip_special_tokens=False),
    }


training_corpus = [
    sample_text,
    comparison_text,
    "Token budgets matter because attention cost grows with sequence length.",
    "Byte-level BPE keeps coverage even for emoji 😊, café, and code like foo_bar().",
    "<|user_start|> Explain why someunknownPlace is not fatal for subword tokenization. <|user_end|>",
]

nanochat_hf = train_nanochat_style_tokenizer(training_corpus, vocab_size=320)
pprint(inspect_hf_tokenizer(nanochat_hf, comparison_text))


{'count': 49,
 'decoded': 'Toronto 2026-04-06: foo_bar() costs 19.99 dollars, and café 😊 is '
            'still open.',
 'ids': [60,
         272,
         292,
         92,
         87,
         229,
         26,
         24,
         26,
         30,
         21,
         24,
         28,
         21,
         24,
         30,
         34,
         303,
         305,
         281,
         276,
         273,
         91,
         229,
         25,
         33,
         22,
         33,
         33,
         229,
         76,
         87,
         290,
         269,
         91,
         20,
         304,
         308,
         307,
         299,
         267,
         92,
         81,
         290,
         229,
         87,
         88,
         265,
         22],
 'tokens': ['T',
            'or',
            'on',
            't',
            'o',
            'Ġ',
            '2',
            '0',
            '2',
            '6',
            '-',
            '0',
            '4'

<a id="side-by-side-comparison"></a>
## Side-by-side comparison


In [13]:
def compare_all(text: str):
    print("TEXT:", text)
    print("\nword-level:")
    pprint(word_level_tokenize(text))
    print("count:", len(word_level_tokenize(text)))

    print("\ncharacter-level:")
    pprint(character_level_tokenize(text))
    print("count:", len(character_level_tokenize(text)))

    print("\nrasbt SimpleTokenizerV2:")
    rasbt_ids = simple_v2.encode(text)
    pprint(rasbt_ids)
    print("decoded:", simple_v2.decode(rasbt_ids))
    print("count:", len(rasbt_ids))

    print("\nGPT-2 tiktoken:")
    pprint(inspect_tiktoken(text))

    print("\nnanochat-style tokenizer:")
    pprint(inspect_hf_tokenizer(nanochat_hf, text))


for text in [
    "In the sunlit terraces of someunknownPlace.",
    comparison_text,
]:
    print("=" * 100)
    compare_all(text)


TEXT: In the sunlit terraces of someunknownPlace.

word-level:
['In', 'the', 'sunlit', 'terraces', 'of', 'someunknownPlace.']
count: 6

character-level:
['I',
 'n',
 ' ',
 't',
 'h',
 'e',
 ' ',
 's',
 'u',
 'n',
 'l',
 'i',
 't',
 ' ',
 't',
 'e',
 'r',
 'r',
 'a',
 'c',
 'e',
 's',
 ' ',
 'o',
 'f',
 ' ',
 's',
 'o',
 'm',
 'e',
 'u',
 'n',
 'k',
 'n',
 'o',
 'w',
 'n',
 'P',
 'l',
 'a',
 'c',
 'e',
 '.']
count: 43

rasbt SimpleTokenizerV2:
[6, 31, 28, 29, 22, 27, 4]
decoded: In the sunlit terraces of someunknownPlace.
count: 7

GPT-2 tiktoken:
{'count': 11,
 'ids': [818, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13],
 'pieces': ['In',
            ' the',
            ' sun',
            'lit',
            ' terr',
            'aces',
            ' of',
            ' some',
            'unknown',
            'Place',
            '.']}

nanochat-style tokenizer:
{'count': 25,
 'decoded': 'In the sunlit terraces of someunknownPlace.',
 'ids': [49,
         86,
         275,
